# XGBoost seat-change classification with expanded predictors

Use every column labelled **Predictor** in `TEST_TRAIN/predictor_descriptions.md`. Identifiers, election metadata and current outcomes are excluded from model inputs. The binary target `seat_changed` is 1 when `winner` differs from `previous_winner`, otherwise 0. Rows missing either party label are excluded. This detects changes between the dataset's party categories, not individual candidates or parties combined within a category such as `oth`.

The original grid (16 combinations), accuracy scoring and fold-local preprocessing are retained. Hyperparameters are selected using expanding-window election cross-validation: train through 2001 and validate on 2005, then train through 2005 and validate on 2010, continuing through 2017. Each validation election has equal weight in mean accuracy. Refit on all eligible rows through 2017, then evaluate on the held-out 2019 election. The classification threshold is fixed at 0.5; the 2024 test.csv is not used.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "TEST_TRAIN" / "train.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")

data = pd.read_csv(ROOT / "TEST_TRAIN" / "train.csv")
election_year = pd.to_numeric(data["election"], errors="raise")
train = data.loc[election_year <= 2017].copy()
test = data.loc[election_year == 2019].copy()
print("Training elections:", sorted(train["election"].unique()))
print("Prediction election: 2019")


In [ ]:
# Read only rows explicitly labelled Predictor; outcomes and metadata stay excluded.
guide = (ROOT / "TEST_TRAIN" / "predictor_descriptions.md").read_text()
FEATURE_COLUMNS = []
for line in guide.splitlines():
    if line.startswith("|"):
        fields = [field.strip() for field in line.strip().strip("|").split("|")]
        if len(fields) >= 2 and fields[1] == "Predictor":
            FEATURE_COLUMNS.append(fields[0].strip("`"))
if not FEATURE_COLUMNS or len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise ValueError("The predictor guide must contain a nonempty, unique predictor list.")

CATEGORICAL_COLUMNS = ["country/region", "previous_winner", "incumbent"]
NUMERIC_COLUMNS = [
    column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS
]

for name, frame in [("train", train), ("test", test)]:
    missing = set(FEATURE_COLUMNS + ["winner"]) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

training = train.dropna(subset=["winner", "previous_winner"]).copy()
testing = test.dropna(subset=["winner", "previous_winner"]).copy()
if training.empty or testing.empty:
    raise ValueError("Training and test data must both contain current and previous winner labels.")

for name, frame, source in [("train", training, train), ("test", testing, test)]:
    frame["seat_changed"] = frame["winner"].ne(frame["previous_winner"]).astype(int)
    print(f"{name}: excluded {len(source) - len(frame)} rows missing target labels; "
          f"{frame['seat_changed'].mean():.3%} changed seats")

display(pd.DataFrame({
    "train_missing_fraction": training[FEATURE_COLUMNS].isna().mean(),
    "test_missing_fraction": testing[FEATURE_COLUMNS].isna().mean(),
}))
print(f"{len(FEATURE_COLUMNS)} predictors; {len(training)} training rows; "
      f"{len(testing)} labelled test rows.")
print("Entirely missing training predictors:",
      training[FEATURE_COLUMNS].columns[
          training[FEATURE_COLUMNS].isna().all()
      ].tolist())


## Model and tuning

XGBoost uses a binary logistic objective for no change (0) versus changed seat (1). Numeric missing values pass through to XGBoost. Categorical predictors are one-hot encoded inside each fold, with unseen categories ignored. `natSW_national_change` and `oth_national_change` are included from the guide but currently have no usable polling signal.

Entire elections stay together: every training row precedes the validation election, and the training window expands after each fold. Accuracy selects the best configuration; because seat changes are less common, the final evaluation also reports change-class precision, recall and F1 alongside an always-no-change baseline.


In [ ]:
preprocessing = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     CATEGORICAL_COLUMNS),
    ("numeric", "passthrough", NUMERIC_COLUMNS),
])
pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=1,
        random_state=42,
    )),
])
# Positional indices refer to the labelled training rows passed to search.fit.
# Election metadata defines the folds but is never a model predictor.
cv_years = pd.to_numeric(training["election"], errors="raise").to_numpy()
if not np.isfinite(cv_years).all():
    raise ValueError("Every training row must have a valid election year.")
validation_years = sorted(year for year in np.unique(cv_years) if year > 2001)
election_splits = []
fold_summary = []
for validation_year in validation_years:
    train_indices = np.flatnonzero(cv_years < validation_year)
    validation_indices = np.flatnonzero(cv_years == validation_year)
    if not len(train_indices):
        raise ValueError(f"No earlier training rows for election {validation_year}.")
    if training.iloc[train_indices]["seat_changed"].nunique() != 2:
        raise ValueError(f"Training before {validation_year} must contain both target classes.")
    election_splits.append((train_indices, validation_indices))
    fold_summary.append({
        "train_through": int(cv_years[train_indices].max()),
        "validation_election": int(validation_year),
        "training_rows": len(train_indices),
        "validation_rows": len(validation_indices),
    })
if len(election_splits) < 2:
    raise ValueError("At least two validation elections after 2001 are required.")
display(pd.DataFrame(fold_summary))

search = GridSearchCV(
    estimator=pipeline,
    param_grid={
        "classifier__n_estimators": [20, 35, 50, 100],
        "classifier__max_depth": [2, 3, 4, 5],
    },
    scoring="accuracy",
    cv=election_splits,
    refit=True,
    n_jobs=-1,
    error_score="raise",
)
search.fit(training[FEATURE_COLUMNS], training["seat_changed"])
model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best mean election CV accuracy: {search.best_score_:.3%}")
split_score_columns = [f"split{i}_test_score" for i in range(len(election_splits))]
display(pd.DataFrame(search.cv_results_)[[
    "param_classifier__n_estimators",
    "param_classifier__max_depth",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
] + split_score_columns].rename(columns={
    column: f"accuracy_{int(year)}"
    for column, year in zip(split_score_columns, validation_years)
}).sort_values("rank_test_score"))


## Held-out seat-change evaluation

Predictions use a fixed probability threshold of 0.5. The confusion matrix and classification report label both binary classes explicitly. The always-no-change baseline shows how much accuracy can be achieved without identifying any changes. Election and constituency names are reporting fields only. Repeated feature or threshold selection using these test results would make 2019 part of model selection.


In [ ]:
change_class_index = list(model.classes_).index(1)
change_probabilities = model.predict_proba(testing[FEATURE_COLUMNS])[:, change_class_index]
predictions = (change_probabilities >= 0.5).astype(int)
actual = testing["seat_changed"]
print(f"Test seat-change accuracy: {accuracy_score(actual, predictions):.3%}")
print(f"Always-no-change baseline: {accuracy_score(actual, np.zeros(len(actual), dtype=int)):.3%}")
class_names = ["No change", "Changed seat"]
print(classification_report(
    actual, predictions, labels=[0, 1], target_names=class_names, zero_division=0,
))
display(pd.DataFrame(
    confusion_matrix(actual, predictions, labels=[0, 1]),
    index=pd.Index(class_names, name="actual"),
    columns=pd.Index(class_names, name="predicted"),
))

report_columns = [
    column for column in ["election", "constituency_name", "previous_winner", "winner", "seat_changed"]
    if column in testing.columns
]
results = testing[report_columns].copy()
results["predicted_seat_changed"] = predictions
results["change_probability"] = change_probabilities
results["correct"] = actual.eq(results["predicted_seat_changed"])
display(results.groupby("election").agg(
    rows=("correct", "size"),
    actual_changes=("seat_changed", "sum"),
    predicted_changes=("predicted_seat_changed", "sum"),
    accuracy=("correct", "mean"),
))
display(results.head(20))


## Feature importance

These are XGBoost's built-in importances for the transformed features. Related polling, swing and share predictors can divide importance between them; these values do not establish causal effects.

In [ ]:
feature_names = model.named_steps["preprocessing"].get_feature_names_out()
importances = pd.Series(
    model.named_steps["classifier"].feature_importances_,
    index=feature_names,
    name="importance",
).sort_values(ascending=False)
display(importances.head(30).to_frame())

assert model.named_steps["preprocessing"].feature_names_in_.tolist() == FEATURE_COLUMNS
assert np.isfinite(model.predict_proba(testing[FEATURE_COLUMNS])).all()
